***api 호출 준비***

In [ ]:
from urllib import parse
import requests
import pandas as pd
from datetime import datetime, timedelta

api_key = 'YOUR_SECRET'

***세트 장비 정보 수집***

In [ ]:
# def search_character(SetItemName, APIKEY):

# 세션 생성
session = requests.Session()

APIKEY = 'YOUR_SECRET'

setItem_df = pd.DataFrame(columns=['setItemId', 'setItemName', 'explatin', 'detailExplain'])


# SetItemName = '세렌디피티'  # 검색할 세트 아이템 이름
SetItemNames = ['세렌디피티', '고대 전장의 발키리', '마력의 영역',
                '무리 사냥의 길잡이', '소울 페어리', '압도적인 자연',
                '에테리얼 오브 아츠', '영원히 이어지는 황금향', '용투장의 난',
                '칠흑의 정화', '한계를 넘어선 에너지', '그림자에 숨은 죽음']  # 검색할 세트 아이템 이름

for SetItemName in SetItemNames:
    # SetItemName_encoded는 URL 인코딩된 세트 아이템 이름
    SetItemName_encoded = parse.quote(SetItemName)

    # URL 조합
    url = f"https://api.neople.co.kr/df/setitems?setItemName={SetItemName_encoded}&limit=100&wordType=full&apikey={APIKEY}"

    # GET 요청
    req = session.get(url)

    setItemId = req.json()['rows'][0]['setItemId']
    setItemName = req.json()['rows'][0]['setItemName']

    url_detail = f'https://api.neople.co.kr/df/setitems/{setItemId}?apikey={APIKEY}'

    req_detail = session.get(url_detail)

    temp_df = pd.DataFrame({
        'setItemId': [setItemId],
        'setItemName': [setItemName],
        'explatin': [req_detail.json()['setItemOption'][0]['explain']],
        'detailExplain': [req_detail.json()['setItemOption'][0]['detailExplain']]
    })

    setItem_df = pd.concat([setItem_df, temp_df], ignore_index=True)

#이거 db에 저장해야함
setItem_df



***집계 대상 유저군 1. 던담 랭커***

In [ ]:
from bs4 import BeautifulSoup as BS
import csv
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import requests


service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

###실제로는 db에서 job_df를 불러와야 함
job_df = pd.read_csv('job_id.csv', encoding='euc-kr')
###


job_df.loc[
    (job_df['jobName'] == '다크나이트') | (job_df['jobName'] == '크리에이터'),
    'jobName'
] = '외전'

job_df_zip = list(zip(job_df['jobName'], job_df['jobGrowName']))

name_list = []
server_list = []
damage_list = []
val_list = []

dundam_df = pd.DataFrame(columns=['name', 'server', 'jobGrowName'])

for jobName, jobGrowName in job_df_zip:
    for page in range(1, 3):
        url = f"https://dundam.xyz/damage_ranking?page={page}&type=7&job={parse.quote(jobGrowName)}&baseJob={parse.quote(jobName)}&weaponType=%EC%A0%84%EC%B2%B4&weaponDetail=%EC%A0%84%EC%B2%B4"
        driver.get(url)

        # 시도는 최대 2번 (원본, 새로고침)
        for attempt in range(2):
            try:
                # nik 요소가 등장할 때까지 최대 10초 대기
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.rk-table.damage .nik"))
                )
            except:
                print(f"[실패] 닉네임 요소 로딩 실패: {jobGrowName}, page {page}, attempt {attempt+1}")
                continue

            name_list = []
            server_list = []

            rank_table = driver.find_element(By.CSS_SELECTOR, "div.rk-table.damage")
            nik_elements = rank_table.find_elements(By.CLASS_NAME, 'nik')
            svname_elements = rank_table.find_elements(By.CLASS_NAME, 'svname')
            name_list += [el.text for el in nik_elements]
            server_list += [el.text for el in svname_elements]

            # 길이 불일치 감지
            if len(name_list) != len(server_list):
                if attempt == 0:
                    print(f"[경고] 불일치: {jobGrowName}, page {page}, 새로고침 후 재시도")
                    driver.refresh()
                    time.sleep(2)  # 새로고침 후 짧게 대기
                    continue   # 새로고침 후 다시 시도
                else:
                    print(f"[실패] 새로고침 후에도 불일치: {jobGrowName}, page {page}")
                    # 데이터 누락 방지 - 에러 로그만 남기고 넘어감
                    break
            else:
                # 정상 추출
                min_len = min(len(name_list), len(server_list))
                if min_len == 0:
                    name_list = [jobGrowName]
                    server_list = ['unknown']
                else:
                    name_list = name_list[:min_len]
                    server_list = server_list[:min_len]

                temp_df = pd.DataFrame({
                    'name': name_list,
                    'server': server_list,
                    'jobGrowName': [jobGrowName] * min_len
                })
                dundam_df = pd.concat([dundam_df, temp_df], ignore_index=True)
                break  # 성공시 반복 종료

driver.quit()
dundam_df = dundam_df.drop_duplicates(subset=['name', 'server', 'jobGrowName'])
print(dundam_df)

dundam_df.to_csv('dundam_damage_ranking.csv', index=False, encoding='utf-8-sig')


**캐릭터 검색용 데이터프레임 생성**

In [ ]:
columns = [
    'serverId', 'characterId', 'characterName', 'level',
    'jobId', 'jobGrowId', 'jobName', 'jobGrowName', 'fame'
]

character_info = pd.DataFrame(columns=columns)

**캐릭터 검색 함수**
*(search_character)*

In [ ]:
def search_character(CharacterName, serverId, APIKEY):
    # 세션 생성
    session = requests.Session()
    
    # CharacterName_encoded는 URL 인코딩된 캐릭터 이름
    CharacterName_encoded = parse.quote(CharacterName)

    # URL 조합
    url = f"https://api.neople.co.kr/df/servers/{serverId}/characters?characterName={CharacterName_encoded}&apikey={APIKEY}"
    
    # GET 요청
    req = session.get(url)

    # 응답 결과 확인
    if req.status_code == 200:
        df = pd.DataFrame(req.json()['rows'])
        return df  # 컬럼 순서 정렬
    else:
        return pd.DataFrame(columns=columns)

**search_character 사용예시**

In [ ]:
from sqlalchemy import create_engine
import pymysql
from sqlalchemy.types import String
from sqlalchemy.types import Text
from sqlalchemy.types import JSON

# MySQL 연결 정보
user = 'root'
password = 'YOUR_SECRET'  # ← 실제 비밀번호로 교체
host = 'localhost'
port = 3306
database = 'bronze'
table_name = 'ranker_info'

# SQLAlchemy 연결 문자열 구성
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# 쿼리 작성
sql = "select u.name, s.serverId from bronze.user_ranker u left outer join bronze.server_info s on u.server = s.serverName"

# 데이터프레임으로 읽기
user_ranker_df = pd.read_sql(sql, engine)

# list(user_ranker_df['name'])

# 검색할 캐릭터 명칭 입력
user_ranker_info = list(zip(user_ranker_df['name'], user_ranker_df['serverId']))

columns = [
    'serverId', 'characterId', 'characterName', 'level',
    'jobId', 'jobGrowId', 'jobName', 'jobGrowName', 'fame'
]

character_info = pd.DataFrame(columns=columns)

for name, serverId in user_ranker_info:
    temp_df = search_character(name, serverId, api_key)
    character_info = pd.concat([character_info, temp_df], ignore_index=True)

character_info = character_info.drop_duplicates(subset=['characterId'])

character_info = character_info[character_info['level'] > 114]

character_info

In [ ]:
# === ranker_info 테이블 생성 + 업서트 ===
import math
from sqlalchemy import create_engine
import pymysql
from sqlalchemy.types import String
from sqlalchemy.types import Text
from sqlalchemy.types import JSON


# MySQL 연결 정보
user = 'root'
password = 'YOUR_SECRET'  # ← 실제 비밀번호로 교체
host = 'localhost'
port = 3306
database = 'bronze'
table_name = 'ranker_info'

# SQLAlchemy 연결 문자열 구성
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

def _ensure_table_ranker_info(db="bronze", table="ranker_info"):
    DDL = f"""
    CREATE TABLE IF NOT EXISTS `{db}`.`{table}` (
      `serverId`       VARCHAR(32)  NOT NULL,
      `characterId`    VARCHAR(64)  NOT NULL,
      `characterName`  VARCHAR(64)  NULL,
      `level`          INT          NULL,
      `jobId`          VARCHAR(64)  NULL,
      `jobGrowId`      VARCHAR(64)  NULL,
      `jobName`        VARCHAR(64)  NULL,
      `jobGrowName`    VARCHAR(64)  NULL,
      `fame`           INT          NULL,
      `inserted_at`    TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,
      PRIMARY KEY (`serverId`, `characterId`),
      KEY `idx_level` (`level`),
      KEY `idx_fame` (`fame`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    with engine.begin() as conn:
        conn.exec_driver_sql(DDL)

def _nan_none(x):
    # pandas NaN -> None
    try:
        return None if (x is None or (isinstance(x, float) and math.isnan(x))) else x
    except Exception:
        return x

def upsert_ranker_info(df, db="bronze", table="ranker_info", batch_size=500):
    """
    character_info DataFrame을 bronze.ranker_info에 업서트.
    필요한 컬럼:
      serverId, characterId, characterName, level, jobId, jobGrowId, jobName, jobGrowName, fame
    """
    required = ['serverId','characterId','characterName','level','jobId','jobGrowId','jobName','jobGrowName','fame']
    for col in required:
        if col not in df.columns:
            raise ValueError(f"DataFrame에 '{col}' 컬럼이 없습니다.")

    _ensure_table_ranker_info(db=db, table=table)

    sql = f"""
    INSERT INTO `{db}`.`{table}`
      (`serverId`,`characterId`,`characterName`,`level`,
       `jobId`,`jobGrowId`,`jobName`,`jobGrowName`,`fame`)
    VALUES
      (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    ON DUPLICATE KEY UPDATE
      `characterName`=VALUES(`characterName`),
      `level`=VALUES(`level`),
      `jobId`=VALUES(`jobId`),
      `jobGrowId`=VALUES(`jobGrowId`),
      `jobName`=VALUES(`jobName`),
      `jobGrowName`=VALUES(`jobGrowName`),
      `fame`=VALUES(`fame`);
    """

    # values 구성
    vals = []
    for _, r in df[required].iterrows():
        vals.append(tuple(_nan_none(r[c]) for c in required))

    inserted = 0
    with engine.begin() as conn:
        for i in range(0, len(vals), batch_size):
            chunk = vals[i:i+batch_size]
            conn.exec_driver_sql(sql, chunk)
            inserted += len(chunk)
    print(f"[DONE] upsert rows: {inserted}")
    return inserted

# === 실행 ===
# character_info DataFrame이 이미 준비되어 있다고 가정(질문에 제공된 코드 바로 아래에서 실행)
upsert_ranker_info(character_info, db="bronze", table="ranker_info")


In [ ]:
# print(character_info.groupby(['jobName','jobGrowName']).size().sort_values(ascending=False))

character_info[character_info['jobGrowName'] == '眞 스핏파이어']

***캐릭터 '기본 정보' 조회 함수***
*(fetch_character)*

In [ ]:
def fetch_character(CharacterName, APIKEY):
    session = requests.Session()

    # 해당 캐릭터 이름과 일치하는 모든 row 추출
    filtered_info = character_info[character_info['characterName'] == CharacterName]

    # characterId와 serverId 쌍 추출
    pairs = list(zip(filtered_info['characterId'], filtered_info['serverId']))

    results = []  # 결과 저장용 리스트

    for characterId, serverId in pairs:
        url = f"https://api.neople.co.kr/df/servers/{serverId}/characters/{characterId}?apikey={APIKEY}"
        req = session.get(url)

        if req.status_code == 200:
            df = pd.DataFrame([req.json()])  # 하나의 dict이므로 []로 감싸서 DataFrame 생성
            results.append(df)
        else:
            print(f"Error fetching character data for {characterId} on {serverId}: {req.status_code}")

    # 여러 요청 결과를 하나의 DataFrame으로 합치기
    if results:
        return pd.concat(results, ignore_index=True)
    else:
        return None

# 호출 예시
fetch_character('캐릭터명', api_key)
   

***캐릭터 '장착 장비' 조회 함수***
*(equip_item)*

***캐릭터 '타임라인' 조회 함수 (character_timeline)***

In [ ]:
from datetime import datetime, timedelta
import requests
import pandas as pd

def character_timeline(CharacterName,
                       startDate=None,
                       endDate=None,
                       APIKEY=None):
    session = requests.Session()

    # 안전한 기본값 처리
    if endDate is None:
        endDate = datetime.now().strftime('%Y-%m-%d %H:%M')
    if startDate is None:
        startDate = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d %H:%M')

    # 해당 캐릭터 이름과 일치하는 모든 row 추출
    filtered_info = character_info[character_info['characterName'] == CharacterName]
    pairs = list(zip(filtered_info['characterId'], filtered_info['serverId']))

    results = []

    for characterId, serverId in pairs:
        url = f"https://api.neople.co.kr/df/servers/{serverId}/characters/{characterId}/timeline?startDate={startDate}&endDate={endDate}&apikey={APIKEY}"
        print(url)
        req = session.get(url)

        if req.status_code == 200:
            df = pd.DataFrame([req.json()])
            results.append(df)
        else:
            print(f"Error fetching character data for {characterId} on {serverId}: {req.status_code}")

    if results:
        return pd.concat(results, ignore_index=True)
    else:
        return None

# 호출 예시
api_key = 'YOUR_SECRET'
df = character_timeline('캐릭터명', APIKEY=api_key)
print(df)
# df.to_csv('character_timeline.csv', index=False)



# for name in character_names:
#     temp_df = character_timeline(name, api_key)
#     character_info = pd.concat([character_info, temp_df], ignore_index=True)

# character_info = character_info.drop_duplicates(subset=['characterId'])

# character_info = character_info[character_info['level'] > 114]

# character_info


In [ ]:
# === ranker_info → timeline_ranker 수집/적재 (Flat 스키마) ===
import json, hashlib, time
from datetime import datetime, timedelta
from typing import Dict, Any, List, Optional
import pandas as pd
from urllib import parse
import requests
from sqlalchemy import create_engine

# MySQL 연결 정보
# user = 'root'
# password = 'YOUR_SECRET'  # ← 실제 비밀번호로 교체
user = 'airflow'
password = "YOUR_SECRET"  # ← 실제 비밀번호로 교체
host = 'localhost'
port = 3306
database = 'bronze'
table_name = 'timeline_ranker'

# SQLAlchemy 연결 문자열 구성
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# 1) timeline_ranker 테이블 보장 (없으면 생성)
def ensure_timeline_table_flat(db_name="bronze", table_name="timeline_ranker"):
    DDL = f"""
    CREATE TABLE IF NOT EXISTS `{db_name}`.`{table_name}` (
      `event_uid`        CHAR(64) PRIMARY KEY,              -- SHA256(characterId|date|code|name|itemId)
      `server_id`        VARCHAR(32)   NOT NULL,
      `character_id`     VARCHAR(64)   NOT NULL,
      `character_name`   VARCHAR(64)   NULL,
      `level`            INT           NULL,
      `job_id`           VARCHAR(64)   NULL,
      `job_grow_id`      VARCHAR(64)   NULL,
      `job_name`         VARCHAR(64)   NULL,
      `job_grow_name`    VARCHAR(64)   NULL,
      `fame`             INT           NULL,
      `adventure_name`   VARCHAR(64)   NULL,
      `guild_id`         VARCHAR(64)   NULL,
      `guild_name`       VARCHAR(64)   NULL,

      `event_date`       DATETIME      NOT NULL,
      `event_code`       INT           NOT NULL,
      `event_name`       VARCHAR(255)  NOT NULL,

      `item_id`          VARCHAR(64)   NULL,
      `item_name`        VARCHAR(255)  NULL,
      `item_rarity`      VARCHAR(32)   NULL,
      `channel_name`     VARCHAR(64)   NULL,
      `channel_no`       INT           NULL,
      `dungeon_name`     VARCHAR(128)  NULL,
      `mist_gear`        TINYINT(1)    NULL,

      `raw_json`         JSON          NULL,
      `inserted_at`      TIMESTAMP     NOT NULL DEFAULT CURRENT_TIMESTAMP,

      KEY `idx_char_date` (`character_id`, `event_date`),
      KEY `idx_server_char` (`server_id`, `character_id`),
      KEY `idx_code_date` (`event_code`, `event_date`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    with engine.begin() as conn:
        conn.exec_driver_sql(DDL)

# 2) 타임라인 payload(네가 준 구조) → 납작화
def _flatten_timeline_payload(payload: Dict[str, Any], meta_override: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    meta = {
        "server_id":      payload.get("serverId"),
        "character_id":   payload.get("characterId"),
        "character_name": payload.get("characterName"),
        "level":          payload.get("level"),
        "job_id":         payload.get("jobId"),
        "job_grow_id":    payload.get("jobGrowId"),
        "job_name":       payload.get("jobName"),
        "job_grow_name":  payload.get("jobGrowName"),
        "fame":           payload.get("fame"),
        "adventure_name": payload.get("adventureName"),
        "guild_id":       payload.get("guildId"),
        "guild_name":     payload.get("guildName"),
    }
    # 랭커 메타가 응답에 없을 때 ranker_info 값으로 보강
    if meta_override:
        for k, v in meta_override.items():
            if meta.get(k) in (None, "", 0) and v is not None:
                meta[k] = v

    rows = []
    tl = payload.get("timeline") or {}
    for ev in tl.get("rows", []):
        code = ev.get("code")
        name = ev.get("name")
        date_str = ev.get("date")

        d = ev.get("data") or {}
        item_id = d.get("itemId")
        item_name = d.get("itemName")
        item_rarity = d.get("itemRarity")
        channel_name = d.get("channelName")
        channel_no = d.get("channelNo")
        dungeon_name = d.get("dungeonName")
        mist_gear = d.get("mistGear")

        uid_base = f"{meta['character_id']}|{date_str or ''}|{code}|{name}|{item_id or ''}"
        event_uid = hashlib.sha256(uid_base.encode("utf-8")).hexdigest()

        # 'YYYY-MM-DD HH:MM' or 'YYYY-MM-DD HH:MM:SS'
        event_dt = None
        if date_str:
            for fmt in ("%Y-%m-%d %H:%M", "%Y-%m-%d %H:%M:%S"):
                try:
                    event_dt = datetime.strptime(date_str, fmt)
                    break
                except Exception:
                    pass

        rows.append({
            **meta,
            "event_uid":   event_uid,
            "event_date":  event_dt,
            "event_code":  code,
            "event_name":  name,
            "item_id":     item_id,
            "item_name":   item_name,
            "item_rarity": item_rarity,
            "channel_name":channel_name,
            "channel_no":  channel_no,
            "dungeon_name":dungeon_name,
            "mist_gear":   mist_gear,
            "raw_json":    json.dumps(ev, ensure_ascii=False),
        })
    return rows

# 3) 업서트
def upsert_timeline_flat(rows: List[Dict[str, Any]], db_name="bronze", table_name="timeline_ranker", batch_size=500) -> int:
    if not rows:
        return 0

    cols = [
        "event_uid","server_id","character_id","character_name","level",
        "job_id","job_grow_id","job_name","job_grow_name","fame",
        "adventure_name","guild_id","guild_name",
        "event_date","event_code","event_name",
        "item_id","item_name","item_rarity","channel_name","channel_no",
        "dungeon_name","mist_gear","raw_json"
    ]
    placeholders = ",".join(["%s"] * len(cols))
    sql = f"""
    INSERT INTO `{db_name}`.`{table_name}` ({",".join(f"`{c}`" for c in cols)})
    VALUES ({placeholders})
    ON DUPLICATE KEY UPDATE
      character_name=VALUES(character_name),
      level=VALUES(level),
      job_id=VALUES(job_id),
      job_grow_id=VALUES(job_grow_id),
      job_name=VALUES(job_name),
      job_grow_name=VALUES(job_grow_name),
      fame=VALUES(fame),
      adventure_name=VALUES(adventure_name),
      guild_id=VALUES(guild_id),
      guild_name=VALUES(guild_name),
      item_id=VALUES(item_id),
      item_name=VALUES(item_name),
      item_rarity=VALUES(item_rarity),
      channel_name=VALUES(channel_name),
      channel_no=VALUES(channel_no),
      dungeon_name=VALUES(dungeon_name),
      mist_gear=VALUES(mist_gear),
      raw_json=VALUES(raw_json)
    """
    vals = [tuple(r.get(c) for c in cols) for r in rows]

    affected = 0
    with engine.begin() as conn:
        for i in range(0, len(vals), batch_size):
            conn.exec_driver_sql(sql, vals[i:i+batch_size])
            affected += len(vals[i:i+batch_size])
    return affected

# 4) 단일 캐릭터: 페이지네이션(next) 포함 수집 → 납작화 → 적재
def fetch_and_store_one(serverId: str, characterId: str,
                        startDate: Optional[str] = None,
                        endDate: Optional[str] = None,
                        code: Optional[str] = None,
                        meta_override: Optional[Dict[str, Any]] = None,
                        db_name="bronze", table_name="timeline_ranker",
                        req_timeout=20, sleep_sec=0.12) -> int:
    ensure_timeline_table_flat(db_name=db_name, table_name=table_name)

    # 기본 기간: 7월 전체(요구에 맞게 기본값 지정) — 필요시 파라미터로 덮어쓰기
    if endDate is None:
        endDate = "2025-07-31 23:59"
    if startDate is None:
        startDate = "2025-07-01 00:00"

    url = f"https://api.neople.co.kr/df/servers/{serverId}/characters/{characterId}/timeline"
    total = 0
    next_token = None

    while True:
        params = {
            "limit": 100,
            "startDate": startDate,
            "endDate": endDate,
            "apikey": api_key
        }
        if code:
            params["code"] = code
        if next_token:
            params["next"] = next_token

        r = session.get(url, params=params, timeout=req_timeout)
        if r.status_code == 404:
            break
        r.raise_for_status()
        payload = r.json() or {}

        rows = _flatten_timeline_payload(payload, meta_override=meta_override)
        n = upsert_timeline_flat(rows, db_name=db_name, table_name=table_name)
        total += n

        # next 토큰: 샘플 구조 기준 timeline.next (문자열 또는 {token:...})
        nxt = payload.get("timeline", {}).get("next")
        next_token = (nxt or {}).get("token") if isinstance(nxt, dict) else nxt
        if not next_token:
            break

        time.sleep(sleep_sec)
    return total

# 5) ranker_info 전체 수행: (serverId, characterId) 기준으로 일괄 수집/적재
from datetime import datetime, timedelta
from typing import Optional
import time
import pandas as pd

def run_timelines_from_ranker_info(db_name="bronze", ranker_table="ranker_info",
                                   out_table="timeline_ranker",
                                   startDate: Optional[str] = None,
                                   endDate: Optional[str] = None,
                                   code: Optional[str] = None,
                                   verbose_every=20) -> int:
    """
    ranker_info 테이블에서 캐릭터 정보를 읽어와 타임라인 수집/적재
    startDate, endDate가 None이면:
        endDate = 현재 시각
        startDate = 현재 시각 - 30일
    """
    # startDate, endDate 기본값 처리
    if endDate is None:
        endDate = datetime.now().strftime('%Y-%m-%d %H:%M')
    if startDate is None:
        startDate = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d %H:%M')

    ensure_timeline_table_flat(db_name=db_name, table_name=out_table)

    # ranker_info 조회
    q = f"""
    SELECT serverId, characterId, characterName, level, jobId, jobGrowId, jobName, jobGrowName, fame
    FROM `{db_name}`.`{ranker_table}`
    WHERE serverId IS NOT NULL AND characterId IS NOT NULL
    """
    rankers = pd.read_sql(q, con=engine)

    if rankers.empty:
        print(f"[WARN] {db_name}.{ranker_table} 에 데이터가 없습니다.")
        return 0

    # 중복 제거
    rankers = rankers.drop_duplicates(subset=["serverId", "characterId"])

    total = 0
    for i, r in enumerate(rankers.itertuples(index=False), start=1):
        serverId = getattr(r, "serverId")
        characterId = getattr(r, "characterId")
        meta = {
            "server_id": serverId,
            "character_id": characterId,
            "character_name": getattr(r, "characterName", None),
            "level": getattr(r, "level", None),
            "job_id": getattr(r, "jobId", None),
            "job_grow_id": getattr(r, "jobGrowId", None),
            "job_name": getattr(r, "jobName", None),
            "job_grow_name": getattr(r, "jobGrowName", None),
            "fame": getattr(r, "fame", None),
        }

        try:
            n = fetch_and_store_one(
                serverId=serverId, characterId=characterId,
                startDate=startDate, endDate=endDate, code=code,
                meta_override=meta,
                db_name=db_name, table_name=out_table
            )
            total += n
            if i % max(1, verbose_every) == 0:
                print(f"[{i}/{len(rankers)}] {serverId}/{characterId} → upsert {n} (누적 {total})")
        except Exception as e:
            print(f"[ERROR] {serverId}/{characterId}: {e}")
            continue
        time.sleep(0.1)

    print(f"[DONE] 총 {total}개 이벤트 upsert 완료")
    return total



***session초기화***

In [ ]:
# 전역 session 초기화 (재시도 포함)
import requests
try:
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
except Exception:
    HTTPAdapter = None
    Retry = None

def make_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    })
    if HTTPAdapter and Retry:
        retry = Retry(
            total=5,
            backoff_factor=0.2,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=False  # 모든 메서드에 재시도 허용(urllib3 2.x는 True/False)
        )
        adapter = HTTPAdapter(max_retries=retry)
        s.mount("https://", adapter)
        s.mount("http://", adapter)
    return s

# 전역 세션 객체 (아래 함수들이 이걸 사용)
session = make_session()

# (선택) 간단 확인
# r = session.get(f"https://api.neople.co.kr/df/servers?apikey={api_key}", timeout=10)
# print("neople status:", r.status_code)


***타임라인 정보 조회 후 db 적재***

In [ ]:
run_timelines_from_ranker_info(startDate="2025-12-13 00:00", endDate="2025-12-14 00:00")
# 필요하면 코드 필터:
# run_timelines_from_ranker_info(code="505")
# 기간 커스텀:
# run_timelines_from_ranker_info(startDate="2025-07-10 00:00", endDate="2025-07-20 23:59")
